[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/09_data_aggregation/09_data_aggregation_practice_quiz.ipynb)

# 09 — Data Aggregation: Practice Quiz

These questions cover the full module 09 toolkit: multi-key groupby, `agg()`, `transform()`, `filter()`, and `pivot_table()`. All questions use the Gapminder dataset.

For multiple-choice questions, write your answer in the provided cell. For code questions, write and run the code. Reveal each solution cell to check your work.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_parquet("gapminder.parquet")
df2007 = df[df["year"] == 2007].copy().reset_index(drop=True)
df.head()

---
### Question 1

The Gapminder dataset has 1,704 rows, 142 countries, and 12 years. What does `df.groupby("continent")["country"].nunique()` return?

A. The number of rows in each continent  
B. The number of distinct countries in each continent  
C. The number of years each continent was observed  
D. The total population of each continent

**Your answer:**

In [ ]:
# Write your answer: A, B, C, or D
answer = ""

In [ ]:
#@title Solution
# Answer: B
# nunique() counts distinct values. groupby("continent")["country"].nunique() gives
# the number of distinct country names within each continent group.
# size() would give the number of rows (142 countries x 12 years = 624 for Africa).
df.groupby("continent")["country"].nunique().sort_values(ascending=False)

---
### Question 2

What is the output type of `df.groupby(["continent", "year"])["lifeExp"].mean()`?

A. A DataFrame with two columns  
B. A Series with a MultiIndex  
C. A Series with a RangeIndex  
D. A scalar (a single number)

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# Grouping by two keys produces a MultiIndex with one level per key.
# Selecting a single column and calling a single aggregation returns a Series.
result = df.groupby(["continent", "year"])["lifeExp"].mean()
print(type(result))
print(type(result.index))

---
### Question 3

You have a MultiIndex Series `s` from a two-key groupby. Which call converts it to a plain DataFrame with `continent` and `year` as regular columns?

A. `s.unstack("year")`  
B. `s.reset_index()`  
C. `s.flatten()`  
D. `pd.DataFrame(s)`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# reset_index() converts every level of the MultiIndex into a regular column.
# unstack() pivots one level into columns (still a MultiIndex on the other axis).
# pd.DataFrame(s) preserves the MultiIndex as the index.
s = df.groupby(["continent", "year"])["lifeExp"].mean()
flat = s.reset_index()
print(flat.dtypes)
flat.head()

---
### Question 4

In the named-aggregation syntax, what does this call return?

```python
df2007.groupby("continent").agg(
    avg_life=("lifeExp", "mean"),
    n_countries=("country", "nunique")
)
```

A. A Series with the mean life expectancy per continent  
B. A DataFrame with one column named `lifeExp` and one named `country`  
C. A DataFrame with one column named `avg_life` and one named `n_countries`  
D. An error, because you cannot mix source columns in one `agg()` call

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: C
# The keyword on the left of the = becomes the output column name.
# Named aggregation allows mixing source columns and functions freely.
df2007.groupby("continent").agg(
    avg_life=("lifeExp", "mean"),
    n_countries=("country", "nunique")
).round(1)

---
### Question 5

What does `df2007.groupby("continent")["lifeExp"].transform("mean")` return?

A. A Series of length 5 (one mean per continent)  
B. A Series of length 142 (one value per row), with every country in the same continent sharing the same value  
C. A DataFrame with continent means in one column and country values in another  
D. The same result as `df2007.groupby("continent")["lifeExp"].agg("mean")`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# transform() returns a result with the same shape as the input (142 rows).
# Each row receives its group's statistic. agg() collapses to 5 rows.
result = df2007.groupby("continent")["lifeExp"].transform("mean")
print("Length:", len(result))
# All Asian countries share the same value
df2007[df2007["continent"] == "Asia"][["country", "lifeExp"]].assign(
    continent_mean=result
).head(5).round(2)

---
### Question 6

You want to add a column `life_above_continent` that is `True` for every country whose life expectancy exceeds its continent's mean in 2007. Which code correctly computes this?

A. `df2007["life_above_continent"] = df2007["lifeExp"] > df2007.groupby("continent")["lifeExp"].agg("mean")`  
B. `df2007["life_above_continent"] = df2007["lifeExp"] > df2007.groupby("continent")["lifeExp"].transform("mean")`  
C. `df2007["life_above_continent"] = df2007["lifeExp"] > df2007["lifeExp"].mean()`  
D. `df2007["life_above_continent"] = df2007.groupby("continent")["lifeExp"].filter(lambda g: g > g.mean())`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# transform() returns a Series aligned to the original index, so the subtraction
# works row by row. agg() returns a 5-row Series indexed by continent name --
# comparing it to a 142-row Series does not align correctly.
# Option C compares to the global mean, not each country's continental mean.
df_check = df2007.copy()
df_check["life_above_continent"] = (
    df_check["lifeExp"] > df_check.groupby("continent")["lifeExp"].transform("mean")
)
df_check.groupby("continent")["life_above_continent"].sum()

---
### Question 7

What does `groupby().filter()` do when a group fails the condition?

A. It sets all values in that group to `NaN`  
B. It drops every row from that group  
C. It drops only the rows within the group that individually fail the condition  
D. It raises a `ValueError`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: B
# filter() is an all-or-nothing operation at the group level.
# If the lambda returns False for a group, every row in that group is dropped.
# This is what distinguishes it from row-level boolean indexing.
kept = df2007.groupby("continent").filter(lambda g: g["gdpPercap"].median() > 10000)
print("Continents remaining:", kept["continent"].unique().tolist())
print("Row count:", len(kept))

---
### Question 8

You want to keep only countries where the life expectancy was above 60 in **every** year from 1977 onward. Which call does this correctly?

A. `df[df["year"] >= 1977].groupby("country").filter(lambda g: g["lifeExp"].min() > 60)`  
B. `df[df["year"] >= 1977].groupby("country").filter(lambda g: g["lifeExp"].mean() > 60)`  
C. `df[df["year"] >= 1977][df["lifeExp"] > 60]`  
D. `df[df["year"] >= 1977].groupby("country").agg(lambda g: g["lifeExp"].min() > 60)`

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: A
# min() > 60 checks that EVERY row in the group is above 60.
# mean() > 60 would pass a country whose average is above 60 but has some years below.
# Option C is row-level and keeps individual rows, not whole countries.
# Option D uses agg(), which returns a summary -- not the original rows.
df_recent = df[df["year"] >= 1977].copy()
consistent = df_recent.groupby("country").filter(lambda g: g["lifeExp"].min() > 60)
print("Countries meeting threshold:", consistent["country"].nunique())

---
### Question 9

What is the key difference between `pd.pivot_table(df, values="lifeExp", index="continent", columns="year", aggfunc="mean")` and `df.groupby(["continent", "year"])["lifeExp"].mean().unstack("year")`?

A. They produce different numeric values  
B. `pivot_table()` adds a `margins=True` argument; `unstack()` does not support margins  
C. They produce the same values; `pivot_table()` states the intent more directly and supports `margins=True` in one call  
D. `unstack()` returns a Series; `pivot_table()` returns a DataFrame

**Your answer:**

In [ ]:
answer = ""

In [ ]:
#@title Solution
# Answer: C
# Both produce identical numeric values. pivot_table() is a single-call interface
# that also supports margins=True and fill_value=. The groupby().unstack() chain
# is more composable when you need named aggregations first.
pt = pd.pivot_table(df, values="lifeExp", index="continent", columns="year", aggfunc="mean")
gb = df.groupby(["continent", "year"])["lifeExp"].mean().unstack("year")
print("Values identical:", pt.round(6).equals(gb.round(6)))

---
### Question 10

Write code to produce a single DataFrame (one row per continent, using `df2007`) with these four columns:
- `median_gdp`: median GDP per capita
- `max_life`: maximum life expectancy
- `min_life`: minimum life expectancy
- `n`: number of countries

Sort by `median_gdp` descending.

In [ ]:
# your code here

In [ ]:
#@title Solution
(
    df2007.groupby("continent").agg(
        median_gdp=("gdpPercap", "median"),
        max_life=("lifeExp",   "max"),
        min_life=("lifeExp",   "min"),
        n        =("country",  "nunique")
    )
    .round(1)
    .sort_values("median_gdp", ascending=False)
)

---
### Question 11

Write code that adds a column `gdp_rank_in_continent` to `df2007`, giving each country's rank by GDP within its continent (rank 1 = highest GDP). Use `transform()` with a lambda.

> Hint: `pd.Series.rank(ascending=False)` ranks from highest to lowest.

In [ ]:
# your code here

In [ ]:
#@title Solution
df_q11 = df2007.copy()
df_q11["gdp_rank_in_continent"] = (
    df_q11.groupby("continent")["gdpPercap"]
    .transform(lambda g: g.rank(ascending=False))
)
# Verify: rank-1 countries in each continent
df_q11[df_q11["gdp_rank_in_continent"] == 1][["continent", "country", "gdpPercap"]].sort_values("continent")

---
### Question 12

Build a continent × year pivot table of **total population** (not mean) using `pd.pivot_table()`. Add `margins=True` and use `margins_name="Total"`. Round to the nearest million (divide by 1,000,000 first). Display only the last four years (1997, 2002, 2007) plus the Total column.

In [ ]:
# your code here

In [ ]:
#@title Solution
df_millions = df.copy()
df_millions["pop"] = df_millions["pop"] / 1_000_000

pt = pd.pivot_table(
    df_millions,
    values="pop",
    index="continent",
    columns="year",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).round(1)

# Select last four years + Total column
pt[[1997, 2002, 2007, "Total"]]